In [1]:
import numpy as np

def combine_datasets_same_length(file1, file2, out_file):
    d1 = np.load(file1)
    d2 = np.load(file2)

    qs = np.concatenate([d1["qs"], d2["qs"]], axis=0)
    xb = np.concatenate([d1["xb"], d2["xb"]], axis=0)

    # handle idx_b
    if d1["idx_b"].ndim == 1:
        idx_b = d1["idx_b"]
    else:
        idx_b = np.concatenate([d1["idx_b"], d2["idx_b"]], axis=0)

    # lambdas: assume shared
    lambdas = d1["lambdas"]

    np.savez(
        out_file,
        qs=qs,
        xb=xb,
        idx_b=idx_b,
        lambdas=lambdas,
    )

    print(f"Saved combined dataset to {out_file}")

In [2]:
combine_datasets_same_length("train_dataset.npz", "test_dataset.npz", "combined_dataset.npz")

Saved combined dataset to combined_dataset.npz


In [38]:
import numpy as np

def combine_datasets(file1, file2, out_file):
    d1 = np.load(file1)
    d2 = np.load(file2)

    l1 = d1["lambdas"].shape[1] if d1["lambdas"].ndim > 1 else d1["lambdas"].shape[0]
    l2 = d2["lambdas"].shape[1] if d2["lambdas"].ndim > 1 else d2["lambdas"].shape[0]
    max_l = max(l1, l2)

    def process_data(data_dict, current_l, target_l):
        diff = target_l - current_l
        
        # 1. Pad qs, xb and lambdas
        qs = np.pad(data_dict["qs"], ((0, 0), (diff, 0), (0, 0)), mode='edge')
        xb = np.pad(data_dict["xb"], ((0, 0), (diff, 0), (0, 0)), mode='edge')

        lam_pad_width = ((0, 0), (diff, 0)) if data_dict["lambdas"].ndim > 1 else ((diff, 0),)
        lambdas = np.pad(data_dict["lambdas"], lam_pad_width, mode='constant', constant_values=0)

        # 3. Create valid mask: (N_traj, N_lambdas)
        # False for the prepended indices, True for the original ones
        n_traj = data_dict["qs"].shape[0]
        valid_block = np.ones((n_traj, current_l), dtype=bool)
        invalid_block = np.zeros((n_traj, diff), dtype=bool)
        valid = np.concatenate([invalid_block, valid_block], axis=1)

        return qs, xb, lambdas, valid

    # Process both datasets to the same max_l
    qs1, xb1, lam1, v1 = process_data(d1, l1, max_l)
    qs2, xb2, lam2, v2 = process_data(d2, l2, max_l)

    # Combine along the trajectory axis (axis 0)
    qs = np.concatenate([qs1, qs2], axis=0)
    xb = np.concatenate([xb1, xb2], axis=0)
    lambdas = np.concatenate([lam1[None, ...], lam2[None, ...]], axis=0)
    valid = np.concatenate([v1, v2], axis=0)

    # For now only handle same idx_b    
    assert(np.array_equal(d1["idx_b"], d2["idx_b"]))

    np.savez(
        out_file,
        qs=qs,
        xb=xb,
        idx_b=d1["idx_b"],
        lambdas=lambdas,
        valid=valid,
    )
    print(f"Combined {file1} and {file2} into {out_file}.")

combine_datasets("train_dataset_11_pts.npz", "train_dataset_22_pts.npz", "train_dataset2.npz")

Combined train_dataset_11_pts.npz and train_dataset_22_pts.npz into train_dataset2.npz.
